Defining states , transitions and emmisions using dictionaries and lists

In [14]:
import math

states=['Start','E','5','I','End']

transitions={
    'Start':{'Start':0,'E':1,'5':0,'I':0,'End':0},
    'E':{'Start':0,'E':0.9,'5':0.1,'I':0,'End':0},
    '5':{'Start':0,'E':0,'5':0,'I':1,'End':0},
    'I':{'Start':0,'E':0,'5':0,'I':0.9,'End':0.1},
    'End':{'Start':0,'E':0,'5':0,'I':0,'End':0}
}

emissions={
    'Start':{'A':0,'C':0,'G':0,'T':0},
    'E':{'A':0.25,'C':0.25,'G':0.25,'T':0.25},
    '5':{'A':0.05,'C':0,'G':0.95,'T':0},
    'I':{'A':0.4,'C':0.1,'G':0.1,'T':0.4},
    'End':{'A':0,'C':0,'G':0,'T':0}
}

Defining function for calculation of the probability of a given path

In [ ]:
def get_log_prob_of_a_given__path(state_path,emitted_bases):
    probability=transitions['Start'][state_path[0]]*emissions[state_path[0]][emitted_bases[0]]
    prev=state_path[0]

    for i in range(1,len(emitted_bases)):
        prob=transitions[prev][state_path[i]]*emissions[state_path[i]][emitted_bases[i]]
        probability*=prob
        prev=state_path[i]

    # For the last transition to the end state
    probability*=0.1
      
    return math.log(probability)
    

In [16]:
state_path="EEEEEEEEEEEEEEEEEE5IIIIIII"
emitted_bases="CTTCATGTGAAAGCAGACGTAAGTCA"

print(f"Log probability : {get_log_prob_of_a_given__path(state_path,emitted_bases):.2f}")

Log probability : -41.22


Implementation of Viterbi algorithm

dp[i][j] is used for storing the probability of emission of a particular base j (it would direct to one of A,C,G,T) by a particular state i (you would have to first reach this state from one of the 5 previous states)

prev_state[i][j] is used to store the previous state from which we got maximum probabiltiy of entering into the state i and then emitting the base j 

In [ ]:
dp=[]
prev_state=[]

Initializing the two matrices

In [ ]:
for i in range(5):
    row=[]
    for j in range(len(emitted_bases)):
        row.append(-1)
    prev_state.append(row)

for i in range(5):
    start_row=[]
    start_row.append(transitions['Start'][states[i]]*emissions[states[i]][emitted_bases[0]])
    for j in range(len(emitted_bases)-1):
        start_row.append(0)
    dp.append(start_row)

Main function loop from where the dp and prev_state are populated.

For the dp table , the element dp[i][j] will have the maximum probabiltiy of transitioning any one of the 5 states from the previous column to state i and then emitting the required base j from state i.

For the prev_state table , the element prev_state[i][j] will have the state from which we got this maximum probability of dp[i][j]

In [ ]:
for col in range(1, len(emitted_bases)):
    for i in range(5):
        maxi = -1e18
        maxi_index = -1
        for k in range(5):
            prob = dp[k][col-1] * transitions[states[k]][states[i]] * emissions[states[i]][emitted_bases[col]]
            if maxi < prob:
                maxi = prob
                maxi_index = k
        dp[i][col] = maxi
        prev_state[i][col] = maxi_index

Reconstructing the best path using the prev_state table , starting from the last state (which emitted the last base) and going in reverse direction , taking those states which yielded the maximum probability of entering the next state (already in the string)

In [ ]:
maxi = -1e18
index = -1
for i in range(5):
    if maxi < dp[i][len(emitted_bases) - 1]:
        maxi = dp[i][len(emitted_bases) - 1]
        index = i

reverse_best_path = [states[index]]
for i in range(len(emitted_bases) - 1, 0, -1):
    index = prev_state[index][i]
    reverse_best_path.append(states[index])

best_path = ''.join(reverse_best_path[::-1])


print("Best possible path for this base emission pattern : "+best_path)


Best possible path for this base emission pattern : EEEEEEEEEEEEEEEEEEEEEEEEEE
